# Mini-project: Books API + pandas + GitHub

**Lesson:** Data Processing Pipeline + Git Workflow (16 November 2026)  
**Level:** Introductory Data Science  
**Goal:** Search for books using the free Open Library API, make a clean DataFrame, save a CSV, and publish your notebook on GitHub.

**Workflow:** API (JSON) → pandas → clean → CSV → Git commit → Git push.

Work individually or in pairs. **No API key is required.** Make only one API request when running the collection cell; avoid repeatedly running the whole notebook.

**Student:** Allan Antonio Canales Corea

## 0. Install and import libraries

If needed, run `%pip install requests pandas`. Run the notebook in VS Code or Google Colab. If offline, place `sample_books.json` next to this notebook; a sample data fallback is included. The sample is **illustrative**, not live API data.

In [1]:
# Uncomment only if your environment needs them:
# %pip install requests pandas

import json
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display

## 1. Collect data from an API

`requests.get()` sends a GET request. `params` adds the query string safely; `.json()` converts JSON to Python objects. `raise_for_status()` helps detect HTTP errors. The API response has a key named **`docs`** containing a list of book records.

**Try:** Change `SEARCH_TERM` to `machine learning` or `data science`.

In [2]:
URL = "https://openlibrary.org/search.json"
SEARCH_TERM = "data science"  # changed from "python programming"
PARAMS = {
    "q": SEARCH_TERM,
    "fields": "title,author_name,first_publish_year,edition_count",
    "limit": 12,
}

try:
    response = requests.get(
        URL, params=PARAMS, timeout=20,
        headers={"User-Agent": "DS4IntroClass/1.0 (educational notebook)"},
    )
    response.raise_for_status()
    data = response.json()
    source = "LIVE Open Library API"
except requests.RequestException as exc:
    print("API unavailable:", exc)
    print("Loading the illustrative sample_books.json file instead.")
    with open("sample_books.json", encoding="utf-8") as file:
        data = json.load(file)
    source = "OFFLINE SAMPLE (not live search results)"

print("Source:", source)
print("Number of records received:", len(data["docs"]))
print("First record:", data["docs"][0] if data["docs"] else "No records")

Source: LIVE Open Library API
Number of records received: 12
First record: {'author_name': ['Joel Grus'], 'edition_count': 4, 'first_publish_year': 2015, 'title': 'Data science from scratch'}


## 2. Convert JSON to a pandas DataFrame

The response is a dictionary. The book records are in `data["docs"]` (a **list of dictionaries**).

In [3]:
df = pd.DataFrame(data["docs"])
display(df.head())

,author_name,edition_count,first_publish_year,title
0,[Joel Grus],4,2015,Data science from scratch
1,"[Tiffany-Anne Timbers, Trevor Campbell, Meliss...",6,2022,Data Science
2,"[Foster Provost, Tom Fawcett]",5,2013,Data Science for Business
3,[Lillian Pierson],10,2015,Data science for dummies
4,"[Nick Singh, Kevin Huo]",1,2021,Ace the Data Science Interview


## 3. Transform and clean the data

We keep only four columns, convert the author list to readable text, fill missing publication years, and sort the rows. This is the **processing** part of the pipeline.

In [4]:
# Make sure these columns exist, even if a record is missing a field.
for column in ["title", "author_name", "first_publish_year", "edition_count"]:
    if column not in df.columns:
        df[column] = None

books = df[["title", "author_name", "first_publish_year", "edition_count"]].copy()
books["author"] = books["author_name"].apply(
    lambda names: ", ".join(names) if isinstance(names, list) else "Unknown"
)
books["year"] = pd.to_numeric(books["first_publish_year"], errors="coerce")
books["editions"] = pd.to_numeric(books["edition_count"], errors="coerce")
books = books[["title", "author", "year", "editions"]]
books = books.dropna(subset=["title"])
books = books.sort_values("title").reset_index(drop=True)

display(books.head(10))

,title,author,year,editions
0,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021,1
1,Big Data,Mouad Radi,2021,1
2,"Business Intelligence, Analytics, and Data Sci...","Ramesh Sharda, Dursun Delen, Efraim Turban",2016,3
3,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022,6
4,Data Science,Ebonie Slough,2022,1
5,Data Science and Big Data Analytics,EMC Education Services,2014,3
6,Data Science for Business,"Foster Provost, Tom Fawcett",2013,5
7,Data science for dummies,Lillian Pierson,2015,10
8,Data science from scratch,Joel Grus,2015,4
9,Doing Data Science,"Rachel Schutt, Cathy O'Neil",2013,3


## 4. Save a CSV

A CSV is an easy-to-share table. `index=False` avoids adding an unwanted index column. This file is created in your current notebook working directory.

In [5]:
OUTPUT_FILE = "books_clean.csv"
books.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

# Verify that the file was really saved:
display(pd.read_csv(OUTPUT_FILE).head())

Saved: books_clean.csv


,title,author,year,editions
0,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021,1
1,Big Data,Mouad Radi,2021,1
2,"Business Intelligence, Analytics, and Data Sci...","Ramesh Sharda, Dursun Delen, Efraim Turban",2016,3
3,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022,6
4,Data Science,Ebonie Slough,2022,1


### Optional: Download the CSV in Google Colab

Run only if using Colab. In VS Code, locate the CSV in your project folder.

In [6]:
# Only run in Google Colab:
# from google.colab import files
# files.download("books_clean.csv")

## 5. Share with Git and GitHub (VS Code terminal)

**First time only:** Create a repository on GitHub, then clone it locally with `git clone <YOUR-REPO-URL>`. Put this notebook and `sample_books.json` inside the repository folder before opening it in VS Code.

Work in an existing repository folder. **Avoid editing the same notebook simultaneously**; each student can use their own branch.

```bash
git status
git pull origin main
git switch -c books-yourname   # optional: create your own branch
git add Student_Books_API_Git.ipynb books_clean.csv
git commit -m "Add cleaned book data workflow"
git push -u origin books-yourname  # if on your branch
```

If you are working directly on `main` in your own repository, replace the last line with `git push origin main`. In a shared project, open a **Pull Request** on GitHub and let your partner review it. If you cannot push, ask for collaborator access or **fork** the repository.

**Important:** `commit` is local. `push` uploads commits to GitHub. `pull` receives teammates' updates. A pull request is a review request, not a Git command.

## 6. Student challenges

1. Change `SEARCH_TERM` and rerun the notebook. What changed in your DataFrame?
2. Add a filter to keep books published in **2010 or later** (use `books[books["year"] >= 2010]`). Display the result.
3. Save the filtered table as `books_recent.csv`.
4. In the terminal, run `git status`. Commit and push your notebook and CSV.
5. **Pair challenge:** Student A changes the API topic on their own branch; Student B reviews the change in a GitHub Pull Request. Then switch roles.

**Deliverables:** notebook (`.ipynb`), CSV (`.csv`), repository URL, and a screenshot or link to one commit or pull request.


In [7]:
# Challenge: keep books published in 2010 or later and save them
recent = books[books["year"] >= 2010].copy()
display(recent)
recent.to_csv("books_recent.csv", index=False)
print("Recent books saved:", len(recent))

,title,author,year,editions
0,Ace the Data Science Interview,"Nick Singh, Kevin Huo",2021,1
1,Big Data,Mouad Radi,2021,1
2,"Business Intelligence, Analytics, and Data Sci...","Ramesh Sharda, Dursun Delen, Efraim Turban",2016,3
3,Data Science,"Tiffany-Anne Timbers, Trevor Campbell, Melissa...",2022,6
4,Data Science,Ebonie Slough,2022,1
5,Data Science and Big Data Analytics,EMC Education Services,2014,3
6,Data Science for Business,"Foster Provost, Tom Fawcett",2013,5
7,Data science for dummies,Lillian Pierson,2015,10
8,Data science from scratch,Joel Grus,2015,4
9,Doing Data Science,"Rachel Schutt, Cathy O'Neil",2013,3


Recent books saved: 12


## Reflection (my answers)

**1. What changed when I changed `SEARCH_TERM`?** The API returned different books, so the titles, authors, years and edition counts all changed. The columns and the structure of the JSON stayed the same, because the same API always answers in the same format.

**2. Where is the list of records in the API response?** In `data["docs"]`. The top level of the JSON is a dictionary, and the value of `docs` is a list of dictionaries, one per book.

**3. What does `index=False` change in the CSV?** Without it pandas writes an extra first column with the row numbers (0, 1, 2...). With `index=False` the file only contains our real columns.

**4. Difference between Git commit and Git push?** `commit` saves a snapshot of my changes only on my computer. `push` uploads those commits to GitHub so other people can see them.